# Chatbot with tools and User Interface

In [1]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [2]:
load_dotenv()
SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")

assert SARVAM_API_KEY is not None, "Could NOT load SARVAM_API_KEY from .env"

## Instantiate the Sarvam AI Client

In [3]:
from sarvamai import SarvamAI

In [4]:
client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
)

## Use Case Definition

Make the chatbot interact with customers, online helping them:

1. Check the Account Balance
2. Review the list of transactions in their account in the last 24 hours

## Setup the tools

In [5]:
def get_balance(account_number: str):
    if account_number == "001002":
        return "INR 51250"
    else:
        return "INR 4300"

def get_transactions(account_number: str):
    if account_number == "001002":
        return [
            ("Time", "Transaction", "INR"),
            ("11:21 AM", "Debit - XYZ Super Market", "INR 651.50"),
            ("4:45 PM",  "Credit - Refund from PQR Braodband Services", "INR 999"),
        ]
    else:
        return "No transactions in the last 24 hrs in your account"

In [6]:
get_transactions("001002")

[('Time', 'Transaction', 'INR'),
 ('11:21 AM', 'Debit - XYZ Super Market', 'INR 651.50'),
 ('4:45 PM', 'Credit - Refund from PQR Braodband Services', 'INR 999')]

## Definition of tools

In [7]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_balance",
            "description": "Get the balance for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_transactions",
            "description": "Get the transactions for the account number",
            "parameters": {
                "type": "object",
                "properties": {
                    "account_number": {"type": "string", "description": "Account Number"},
                },
                "required": ["account_number"],
            },
        },
    },
]

## chat() to call Sarvam API

In [8]:
def chat(messages: list, tools: list):
    while True:
        response = client.chat.completions(
            model="sarvam-105b-conversations",
            messages=messages,
            tools=tools
        )

        if response.choices[0].finish_reason == "stop":
            break;
        
        # 1. Read list of tool_calls requested by AI and append to chat_history
        tool_calls = response.choices[0].message.tool_calls
        messages.append(
            {
                "role": "assistant",
                "tool_calls": tool_calls
            }
        )
    
        handle_tool_calls(tool_calls, messages)
        
    # 3. Call AI again with updated history to process further 
    
    return response

In [15]:
tools_map = {
    'get_balance': get_balance,
    'get_transactions': get_transactions,
}

def invoke_tool(tool_name: str, tool_args: dict):
    result = tools_map[tool_name](**tool_args)    
    return result

In [16]:
invoke_tool('get_balance', {'account_number': '001002'})

'INR 51250'

In [17]:
invoke_tool('get_transactions', {'account_number': '001002'})

[('Time', 'Transaction', 'INR'),
 ('11:21 AM', 'Debit - XYZ Super Market', 'INR 651.50'),
 ('4:45 PM', 'Credit - Refund from PQR Braodband Services', 'INR 999')]

In [11]:
import json
def handle_tool_calls(tool_calls: list, messages: list):
    # 2. For each tool
    for tool_call in tool_calls:
    
        # a. Get the function name and arguments
        f_name = tool_call.function.name
        f_args = json.loads(tool_call.function.arguments)
        
        # b. Invoke the function and get the result
        result = invoke_tool(f_name, f_args)
        print("Result from Tool Call :: ", result)   
        
        # c. Add the result to chat_history
        messages.append(
            {
                "role": "tool",
                "content": str(result),
                "tool_call_id": tool_call.id
                
            }
        )

In [12]:
messages = [
            {"role": "user", "content": "Get the balance for account 001002"},
        ]

response = chat(messages, tools)
print(response.choices[0].message.content)
#print(response.model_dump_json(indent=2))

Result from Tool Call ::  INR 51250
The balance for account **001002** is **INR 51,250**.


## Chat with the bot

In [ ]:
BLUE_TEXT = "\033[94m"
BLACK_TEXT = "\033[0m"

In [28]:
def run_chatbot(user_message: str, chat_history: list):

    # 1. initialize chat history with System prompt
    # 2. Get user message and add it to chat history
    # 3. Call chat() to get response from AI 
    # 4. Print the response and add it to chat history
    # 5. Repeat steps 2-4 until user types "quit"

    # gradio sends us the chat_history in a slightly different format
    # convert it into a format that Sarvam AI expects
    #      'role': 'assistant', 'metadata': None, 'content': [{'text': 'how are you', 'type': 'text'}]
    chat_history = [
        {
            "role": message["role"], 
            "content": message["content"][0]["text"]
        } for message in chat_history
    ]
    
    # 1. initialize chat history with System prompt
    # Changes: Prepend the system prompt to gradio chat_history
    
    chat_history=[
        {"role": "system", "content": "You are a Customer Service Rep from ABC Bank."},
    ] + chat_history

          
    # 2. Get user message and add it to chat history
    # Changes: user_message sent as input from gradio, so input() is not required.
    #          Looping is taken care of by gradio, hence while loop can be removed.  
    
    chat_history.append(
        {"role": "user", "content": user_message.strip()}
    )

    # 3. Call chat() to get response from AI 
    # Changes: NO changes
    response = chat(chat_history, tools)

    # 4. Print the response and add it to chat history
    # Changes: Return the response, since printing and adding to chat history shall be taken care by gradio.
    
    return response.choices[0].message.content

#### Run the chatbot

In [ ]:
run_chatbot()

### Integrate with gradio

In [29]:
import gradio as gr

chat_interface = gr.ChatInterface(
    fn=run_chatbot, 
    title="ABC Bank - Customer Service Chat", 
)

chat_interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Result from Tool Call ::  INR 51250
Result from Tool Call ::  [('Time', 'Transaction', 'INR'), ('11:21 AM', 'Debit - XYZ Super Market', 'INR 651.50'), ('4:45 PM', 'Credit - Refund from PQR Braodband Services', 'INR 999')]
Result from Tool Call ::  INR 4300


In [30]:
chat_interface.close()

Closing server running on port: 7860
